# QEC syndrome Bronze discovery

This notebook inspects the supplied archive without changing or extracting the Bronze files. It records the source members, schema, samples, quantities, candidate identifiers, and structural checks needed before building Silver.

In [15]:
from ast import literal_eval
from pathlib import Path
from zipfile import ZipFile

import pandas as pd

# JupyterLab uses /course-data. The second path makes this notebook usable outside Docker too.
candidate_archives = [
    Path('/course-data/raw/source=qec_syndromes/syndromes_dataset.zip'),
    Path('../datasets/student-bundle/core/raw/source=qec_syndromes/syndromes_dataset.zip'),
]
archive = next((path for path in candidate_archives if path.exists()), None)
if archive is None:
    raise FileNotFoundError('Could not find the syndrome archive. Run this notebook in the course JupyterLab container.')

print(f'Archive found: {archive}')
print(f'Archive size: {archive.stat().st_size:,} bytes')

Archive found: /course-data/raw/source=qec_syndromes/syndromes_dataset.zip
Archive size: 358,017 bytes


In [16]:
with ZipFile(archive) as bundle:
    members = bundle.infolist()

print(f'Archive members: {len(members)}')
for member in members:
    print(f'- {member.filename}: {member.file_size:,} bytes')

Archive members: 8
- d-3_pfr-0.000010_nb-10M.csv: 4,411 bytes
- d-3_pfr-0.000050_nb-10M.csv: 13,715 bytes
- d-3_pfr-0.000100_nb-10M.csv: 31,115 bytes
- d-3_pfr-0.000500_nb-10M.csv: 89,438 bytes
- d-3_pfr-0.001000_nb-10M.csv: 181,222 bytes
- d-3_pfr-0.005000_nb-10M.csv: 1,323,765 bytes
- d-3_pfr-0.010000_nb-10M.csv: 3,148,485 bytes
- README.txt: 343 bytes


In [17]:
with ZipFile(archive) as bundle:
    csv_members = [member.filename for member in bundle.infolist() if member.filename.endswith('.csv')]
    first_csv = sorted(csv_members)[0]
    with bundle.open(first_csv) as file:
        sample = pd.read_csv(file, nrows=5)

print(f'Sample source: {first_csv}')
print(f'Columns: {list(sample.columns)}')
print('Data types:')
print(sample.dtypes.to_string())
print('First five rows:')
display(sample)

Sample source: d-3_pfr-0.000010_nb-10M.csv
Columns: ['labels', 'syndromes', 'quantity']
Data types:
labels        int64
syndromes    object
quantity      int64
First five rows:


,labels,syndromes,quantity
0,0,"((0, 0, 0, 0), (0, 0, 0, 0), (0, 0, 0, 0), (0,...",9987291
1,0,"((0, 0, 1, 0), (0, 0, 1, 0), (0, 0, 0, 0), (0,...",486
2,1,"((0, 0, 1, 0), (0, 0, 0, 0), (0, 0, 0, 0), (0,...",476
3,0,"((0, 1, 0, 0), (0, 1, 0, 0), (0, 0, 0, 0), (0,...",448
4,0,"((0, 0, 0, 0), (0, 1, 0, 0), (0, 1, 0, 0), (0,...",439


In [18]:
def parse_syndrome(value):
    parsed = literal_eval(value)
    return tuple(tuple(int(bit) for bit in round_values) for round_values in parsed)

profile_rows = []
with ZipFile(archive) as bundle:
    for csv_name in sorted(csv_members):
        with bundle.open(csv_name) as file:
            frame = pd.read_csv(file)
        parsed = frame['syndromes'].map(parse_syndrome)
        shapes = sorted({(len(value), tuple(len(row) for row in value)) for value in parsed})
        profile_rows.append({
            'file': csv_name,
            'rows': len(frame),
            'columns': list(frame.columns),
            'label_values': sorted(frame['labels'].unique().tolist()),
            'quantity_total': int(frame['quantity'].sum()),
            'quantity_min': int(frame['quantity'].min()),
            'quantity_max': int(frame['quantity'].max()),
            'syndrome_shapes': shapes,
            'binary_values_only': all(bit in {0, 1} for value in parsed for row in value for bit in row),
        })

profile = pd.DataFrame(profile_rows)
print('Per-file discovery profile:')
display(profile)

Per-file discovery profile:


,file,rows,columns,label_values,quantity_total,quantity_min,quantity_max,syndrome_shapes,binary_values_only
0,d-3_pfr-0.000010_nb-10M.csv,68,"[labels, syndromes, quantity]","[0, 1]",10000000,1,9987291,"[(4, (4, 4, 4, 4))]",True
1,d-3_pfr-0.000050_nb-10M.csv,215,"[labels, syndromes, quantity]","[0, 1]",10000000,1,9936420,"[(4, (4, 4, 4, 4))]",True
2,d-3_pfr-0.000100_nb-10M.csv,491,"[labels, syndromes, quantity]","[0, 1]",10000000,1,9874116,"[(4, (4, 4, 4, 4))]",True
3,d-3_pfr-0.000500_nb-10M.csv,1407,"[labels, syndromes, quantity]","[0, 1]",10000000,1,9384071,"[(4, (4, 4, 4, 4))]",True
4,d-3_pfr-0.001000_nb-10M.csv,2854,"[labels, syndromes, quantity]","[0, 1]",10000000,1,8806118,"[(4, (4, 4, 4, 4))]",True
5,d-3_pfr-0.005000_nb-10M.csv,20887,"[labels, syndromes, quantity]","[0, 1]",10000000,1,5309458,"[(4, (4, 4, 4, 4))]",True
6,d-3_pfr-0.010000_nb-10M.csv,49676,"[labels, syndromes, quantity]","[0, 1]",10000000,1,2838588,"[(4, (4, 4, 4, 4))]",True


In [19]:
print('Candidate identifiers and decisions:')
print('- experiment_id: filename without .csv, because it identifies the fault-rate experiment.')
print('- source_record_id: qec_syndromes/<filename>#row-<CSV row number>, because syndrome alone is not unique.')
print('- quantity remains a weight; it must not be expanded into repeated rows.')
print('- a syndrome may appear with both labels, so label is part of the aggregate row meaning.')
print('- Silver remains source-specific; qec_syndromes is not joined to Google or QASMBench here.')

Candidate identifiers and decisions:
- experiment_id: filename without .csv, because it identifies the fault-rate experiment.
- source_record_id: qec_syndromes/<filename>#row-<CSV row number>, because syndrome alone is not unique.
- quantity remains a weight; it must not be expanded into repeated rows.
- a syndrome may appear with both labels, so label is part of the aggregate row meaning.
- Silver remains source-specific; qec_syndromes is not joined to Google or QASMBench here.


## Interpretation

The profile above is the discovery evidence. Before writing Silver, investigate any row whose columns, labels, quantity, syndrome shape, or binary-domain check differs from the expected contract. The production parser should repeat these checks and record rejected rows in `results/part1/data_issues.parquet`.